# Smart MCQ Solver Challenge 

The objective of this project is to build machine learning and deep learning models to identify and rank the top three most likely correct answers for multiple-choice questions (MCQs).

---

## Overview

Each question in the dataset consists of a **prompt** and five answer choices: **A, B, C, D,** and **E**.

Instead of predicting a single answer, models generate the **top three answers in ranked order**. A higher rank for the correct answer yields a higher evaluation score.

Three approaches are implemented and evaluated on a shared validation split:
1. **Model 1**: TF-IDF + K-Nearest Neighbors (Retrieval Baseline)
2. **Model 2**: TF-IDF + Logistic Regression (Classifier)
3. **Model 3**: Bi-LSTM with Self-Attention (Neural Network)

---

## Evaluation Metric

Performance is measured using **Mean Average Precision at 3 (MAP@3)**.

For a single question, **Average Precision at 3 (AP@3)** is defined as:

$$
AP@3 = \begin{cases} \frac{1}{r}, & \text{if correct answer is at rank } r \le 3 \\ 0, & \text{otherwise} \end{cases}
$$

where $r$ is the 1-based rank position of the correct answer.

Overall score is the mean across all $N$ questions:

$$
MAP@3 = \frac{1}{N} \sum_{i=1}^{N} AP_i@3
$$

### Example

Assuming the correct answer is **A**:

| Prediction | AP@3 | Explanation |
|---|---|---|
| `A B C` | **1.000** | Rank 1 ($1/1$) |
| `B A C` | **0.500** | Rank 2 ($1/2$) |
| `C D A` | **0.333** | Rank 3 ($1/3$) |
| `B C D` | **0.000** | Not in top 3 |

Top-1 Accuracy and Macro F1-score are also recorded for comparison.

---

## Dataset Structure

| Column | Description |
|---|---|
| `id` | Unique identifier for each question |
| `prompt` | Question text / problem statement |
| `A`, `B`, `C`, `D`, `E` | Five answer choices |
| `answer` | Correct answer label (`A`-`E`) |

In [ ]:
import numpy as np
import pandas as pd
import os

if os.path.exists('/kaggle/input'):
    for dirname, _, filenames in os.walk('/kaggle/input'):
        for filename in filenames:
            print(os.path.join(dirname, filename))

## 1. Imports & Setup

Essential libraries for data handling, machine learning, and deep learning are imported. Fixed random seeds (`42`) are configured across NumPy, Python, and PyTorch for reproducibility.

In [ ]:
import os
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from collections import Counter

# set seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

LABELS = ['A', 'B', 'C', 'D', 'E']
plt.style.use('ggplot')

## 2. Weights & Biases (W&B) Setup

Weights & Biases (W&B) is initialized to log loss, accuracy, Macro F1, and MAP@3 metrics across experiments.

In [ ]:
!pip install -q wandb

import wandb
from kaggle_secrets import UserSecretsClient

wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))

WANDB_PROJECT = "24f2001637-t22026"
print("W&B ready. Project:", WANDB_PROJECT)

## 3. Helper Functions

Utility functions defined for data cleaning, evaluation, and output generation:

- **`clean_text`**: Normalizes case, removes extra whitespace, and filters special characters.
- **`build_option_text`**: Combines prompt and option text with a `[SEP]` separator token.
- **`label_to_idx` / `idx_to_label`**: Maps letter choices (`A`-`E`) to numerical indices (`0`-`4`) and vice versa.
- **`average_precision_at_k` / `map_at_3`**: Computes AP@3 and MAP@3 metrics.
- **`compute_metrics`**: Calculates Top-1 Accuracy, Macro F1, and MAP@3.
- **`format_submission`**: Formats output predictions into `submission.csv`.

In [ ]:
# clean text
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s.,!?;:()\-\'"]+', ' ', text)
    return text.strip()


def build_option_text(prompt, option_text):
    return clean_text(prompt) + " [SEP] " + clean_text(option_text)


def label_to_idx(label):
    return ord(label.upper()) - ord('A')


def idx_to_label(idx):
    return chr(ord('A') + idx)


# calculate AP@3 score
def average_precision_at_k(predicted, actual, k=3):
    predicted = predicted[:k]
    if actual not in predicted:
        return 0.0
    rank = predicted.index(actual) + 1
    return 1.0 / rank


def map_at_3(predictions, actuals):
    scores = [average_precision_at_k(pred, actual, k=3) for pred, actual in zip(predictions, actuals)]
    return float(np.mean(scores))


# compute all metrics
def compute_metrics(true_labels, top3_predictions):
    top1_predictions = [pred[0] for pred in top3_predictions]
    acc = accuracy_score(true_labels, top1_predictions)
    f1 = f1_score(true_labels, top1_predictions, average="macro", labels=LABELS, zero_division=0)
    map3 = map_at_3(top3_predictions, true_labels)
    return {
        "accuracy": acc,
        "f1_macro": f1,
        "map@3": map3
    }


# save predictions
def format_submission(ids, predictions, output_path):
    formatted_preds = []
    for pred in predictions:
        formatted_preds.append(" ".join(pred[:3]))
        
    submission = pd.DataFrame({
        'ID': ids,
        'Prediction': formatted_preds
    })
    submission.to_csv(output_path, index=False)
    print(f"Saved submission to {output_path} ({len(submission)} rows)")
    return submission

## 4. Load Data

Training (`train.csv`) and test (`test.csv`) sets are loaded. Missing option values are replaced with empty strings.

In [ ]:
# load train and test data
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

# fill missing option values
for df in [train_df, test_df]:
    for col in LABELS:
        df[col] = df[col].fillna("").astype(str)

print(f"Train shape: {train_df.shape} | Test shape: {test_df.shape}")
print("\nColumn types:")
print(train_df.dtypes)

train_df.head(3)

## 5. Exploratory Data Analysis (EDA)

Key properties analyzed prior to modeling:
- Target label balance across choices `A`-`E`.
- Prompt and option word length distributions.
- Presence of duplicate question prompts.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

prompt_len = train_df["prompt"].str.split().str.len()
option_lens = {label: train_df[label].str.split().str.len() for label in LABELS}

# answer distribution
answer_counts = train_df["answer"].value_counts().sort_index()
axes[0, 0].bar(answer_counts.index, answer_counts.values)
axes[0, 0].set_title("Answer Label Distribution")
for i, v in enumerate(answer_counts.values):
    axes[0, 0].text(i, v, str(v), ha="center", va="bottom")

# prompt length
axes[0, 1].hist(prompt_len, bins=40)
axes[0, 1].axvline(prompt_len.mean(), linestyle="--")
axes[0, 1].set_title(f"Prompt Length (mean = {prompt_len.mean():.0f} words)")

# option length
for label in LABELS:
    axes[0, 2].hist(option_lens[label], bins=30, alpha=0.5, label=label)
axes[0, 2].legend()
axes[0, 2].set_title("Option Lengths")

# prompt length by answer
for label in LABELS:
    axes[1, 0].hist(prompt_len[train_df["answer"] == label], bins=20, alpha=0.5, label=label)
axes[1, 0].legend()
axes[1, 0].set_title("Prompt Length by Answer")

# total text length
total_len = prompt_len + sum(option_lens.values())
axes[1, 1].hist(total_len, bins=40)
axes[1, 1].set_title(f"Total Text Length (mean = {total_len.mean():.0f} words)")

# duplicate prompts
duplicate_count = train_df.duplicated("prompt", keep=False).sum()
axes[1, 2].bar(["Unique", "Duplicate"], [len(train_df) - duplicate_count, duplicate_count])
axes[1, 2].set_title("Prompt Uniqueness")

plt.tight_layout()
plt.show()

print(f"Duplicate prompts: {duplicate_count} / {len(train_df)}")

## 6. Preprocessing & Train/Validation Split

Text formatting and data splitting:
1. **Text Formatting**: Prompt and options are concatenated into single text strings.
2. **Train/Validation Split**: Data is partitioned into 90% training (`train_sub`) and 10% validation (`val_sub`).

In [ ]:
def build_mcq_text(row, include_opts=True):
    prompt = clean_text(str(row['prompt']))
    if not include_opts:
        return prompt
    opts = []
    for label in LABELS:
        opts.append(f"{label}: {clean_text(str(row[label]))}")
    return prompt + " [SEP] " + " ".join(opts)


# split train/val
train_sub, val_sub = train_test_split(train_df, test_size=0.1, random_state=42)
train_sub = train_sub.reset_index(drop=True)
val_sub = val_sub.reset_index(drop=True)

print(f"Train split: {len(train_sub)} | Validation split: {len(val_sub)}")

val_true = [str(row['answer']) for _, row in val_sub.iterrows()]
all_run_results = []

## 7. Model 1: TF-IDF + K-Nearest Neighbors (Baseline)

A training-free retrieval baseline:
1. TF-IDF vectors (unigrams + bigrams, up to 50,000 features) are constructed.
2. Cosine similarity is computed between validation queries and training samples.
3. Top $k=15$ neighbors vote for answer labels, weighted by similarity score.
4. Top 3 highest-voted choices are selected.

In [ ]:
run = wandb.init(
    project=WANDB_PROJECT,
    name="tfidf-knn-baseline",
    job_type="eval",
    config={"model": "tfidf-knn", "max_features": 50000, "ngram_range": (1, 2), "k_neighbors": 15}
)

print("Fitting TF-IDF vectorizer...")

vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2
)

train_texts = train_sub.apply(build_mcq_text, axis=1)
train_embeddings = vectorizer.fit_transform(train_texts)

print(f"Vocabulary size: {len(vectorizer.vocabulary_):,} | Matrix shape: {train_embeddings.shape}")


# KNN prediction
def tfidf_predict(row, k=15):
    query = vectorizer.transform([build_mcq_text(row)])
    similarities = cosine_similarity(query, train_embeddings).ravel()
    neighbor_idx = np.argsort(similarities)[-k:][::-1]

    votes = {label: 0.0 for label in LABELS}
    for idx in neighbor_idx:
        score = similarities[idx]
        if score < 0.01:
            continue
        neighbor_answer = train_sub.iloc[idx]["answer"].upper()
        if neighbor_answer in votes:
            votes[neighbor_answer] += score

    sorted_labels = sorted(votes, key=votes.get, reverse=True)
    return sorted_labels[:3]


print("Predicting on the validation set...")
tfidf_val_preds = val_sub.apply(tfidf_predict, axis=1).tolist()

tfidf_metrics = compute_metrics(val_true, tfidf_val_preds)
print(f"Run 1 (TF-IDF + KNN) -> accuracy: {tfidf_metrics['accuracy']:.4f} | "
      f"F1 (macro): {tfidf_metrics['f1_macro']:.4f} | MAP@3: {tfidf_metrics['map@3']:.4f}")

wandb.log(tfidf_metrics)
all_run_results.append({
    "run": "tfidf-knn-baseline",
    "accuracy": tfidf_metrics["accuracy"],
    "f1_macro": tfidf_metrics["f1_macro"],
    "map@3": tfidf_metrics["map@3"]
})
wandb.finish()

## 8. Model 2: TF-IDF + Logistic Regression

A multi-class linear classifier approach:
1. TF-IDF feature matrices are extracted from text blocks.
2. A multi-class Logistic Regression model is fitted on the training split.
3. Class probability estimates are obtained for options `A`-`E`.
4. Options are ranked by probability to select top 3 predictions.

In [ ]:
run = wandb.init(
    project=WANDB_PROJECT,
    name="tfidf-logistic-regression",
    job_type="eval",
    config={"model": "tfidf-logreg", "max_features": 50000, "C": 1.0, "max_iter": 1000}
)

print("Training Logistic Regression classifier on TF-IDF features...")

# train logistic regression
logreg = LogisticRegression(max_iter=1000, C=1.0)
logreg.fit(train_embeddings, train_sub["answer"])

val_texts = val_sub.apply(build_mcq_text, axis=1)
val_embeddings = vectorizer.transform(val_texts)

val_probs = logreg.predict_proba(val_embeddings)
class_order = list(logreg.classes_)

# get top 3 predictions
logreg_val_preds = []
for i in range(len(val_probs)):
    probs_i = val_probs[i]
    ranked = sorted(class_order, key=lambda label: probs_i[class_order.index(label)], reverse=True)
    logreg_val_preds.append(ranked[:3])

logreg_metrics = compute_metrics(val_true, logreg_val_preds)
print(f"Run 2 (TF-IDF + LogReg) -> accuracy: {logreg_metrics['accuracy']:.4f} | "
      f"F1 (macro): {logreg_metrics['f1_macro']:.4f} | MAP@3: {logreg_metrics['map@3']:.4f}")

wandb.log(logreg_metrics)
all_run_results.append({
    "run": "tfidf-logistic-regression",
    "accuracy": logreg_metrics["accuracy"],
    "f1_macro": logreg_metrics["f1_macro"],
    "map@3": logreg_metrics["map@3"]
})
wandb.finish()

## 9. Model 3: Bi-LSTM with Self-Attention

A PyTorch deep neural network architecture:
1. Prompt text is paired with each answer option.
2. A shared Bidirectional LSTM encodes input sequences.
3. A Self-Attention layer aggregates hidden states into fixed-length vectors.
4. A scoring network assigns logits to each option.

### 9.1 Build Vocabulary

A vocabulary of the top 15,000 frequent words is constructed from the training set (`<PAD>=0`, `<UNK>=1`).

In [ ]:
# build vocab
all_training_texts = []
for _, row in train_sub.iterrows():
    prompt = clean_text(str(row['prompt']))
    for label in LABELS:
        all_training_texts.append(build_option_text(prompt, str(row[label])))

word_counts = Counter()
for text in all_training_texts:
    for word in text.split():
        word_counts[word] += 1

VOCAB = {"<PAD>": 0, "<UNK>": 1}
for word, _ in word_counts.most_common(15000 - 2):
    VOCAB[word] = len(VOCAB)

VOCAB_SIZE = len(VOCAB)
MAX_LEN = 128
print(f"Vocabulary size: {VOCAB_SIZE:,}")


def encode(text, max_len=MAX_LEN):
    words = text.lower().split()[:max_len]
    ids = [VOCAB.get(w, 1) for w in words]
    padding = [0] * (max_len - len(ids))
    return ids + padding

### 9.2 Dataset & Model Architecture

- **`MCQDataset`**: Formats prompt-option pairs into token index tensors (`MAX_LEN = 128`).
- **`SelfAttn`**: Computes token-level attention weights for summary vector generation.
- **`BiLSTMModel`**: Combines embedding, Bi-LSTM, self-attention, and option scoring layers.

In [ ]:
class MCQDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
        self.has_answer = 'answer' in df.columns

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        prompt = clean_text(str(row['prompt']))
        
        encoded_opts = []
        for label in LABELS:
            text = build_option_text(prompt, str(row[label]))
            encoded_opts.append(encode(text))
            
        options = torch.tensor(encoded_opts, dtype=torch.long)
        
        if self.has_answer:
            label = torch.tensor(label_to_idx(str(row['answer'])), dtype=torch.long)
        else:
            label = torch.tensor(-1, dtype=torch.long)
            
        return options, label


class SelfAttn(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn_score = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x, mask=None):
        scores = self.attn_score(x).squeeze(-1)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        weights = F.softmax(scores, dim=-1).unsqueeze(-1)
        return (x * weights).sum(dim=1)


class BiLSTMModel(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hidden_dim=256, num_layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            emb_dim, hidden_dim, num_layers,
            batch_first=True, bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.attention = SelfAttn(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.scorer = nn.Sequential(
            nn.Linear(hidden_dim * 2, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )

    def encode_option(self, x):
        mask = (x != 0)
        lstm_out, _ = self.lstm(self.dropout(self.embedding(x)))
        return self.dropout(self.attention(lstm_out, mask))

    def forward(self, options):
        batch_size, num_options, seq_len = options.shape
        encoded = self.encode_option(options.view(batch_size * num_options, seq_len))
        return self.scorer(encoded).view(batch_size, num_options)


print("BiLSTM model defined.")

### 9.3 Model Training

The Bi-LSTM model is trained for 12 epochs using AdamW optimizer (`lr = 3e-4`) and gradient clipping. Training loss and validation metrics (Accuracy, Macro F1, MAP@3) are evaluated per epoch.

In [ ]:
def run_training(model, train_df, val_df, epochs=12, batch_size=64, lr=3e-4, name="model"):
    train_loader = DataLoader(MCQDataset(train_df), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(MCQDataset(val_df), batch_size=batch_size, shuffle=False)

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    history = []
    best_map3 = 0.0
    final_val_preds = None

    for epoch in range(1, epochs + 1):

        # training
        model.train()
        total_loss, correct, samples = 0, 0, 0

        for options, labels in train_loader:
            options = options.to(DEVICE)
            labels = labels.to(DEVICE)

            optimizer.zero_grad()
            logits = model(options)
            loss = criterion(logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            batch_n = labels.size(0)
            total_loss += loss.item() * batch_n
            correct += (logits.argmax(1) == labels).sum().item()
            samples += batch_n

        train_loss = total_loss / samples
        train_acc = correct / samples

        # validation
        model.eval()
        logits_list = []
        with torch.no_grad():
            for options, _ in val_loader:
                logits = model(options.to(DEVICE))
                logits_list.append(logits.cpu().numpy())

        logits = np.concatenate(logits_list)
        
        val_preds = []
        for row in logits:
            sorted_indices = np.argsort(row)[::-1][:3]
            top3 = [idx_to_label(i) for i in sorted_indices]
            val_preds.append(top3)
            
        final_val_preds = val_preds
        epoch_metrics = compute_metrics(val_true, val_preds)

        print(f"[{name}] E{epoch:02d} | TrainLoss={train_loss:.4f} | "
              f"TrainAcc={train_acc:.4f} | ValAcc={epoch_metrics['accuracy']:.4f} | "
              f"ValF1={epoch_metrics['f1_macro']:.4f} | ValMAP3={epoch_metrics['map@3']:.4f}")

        wandb.log({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_acc,
            "val_accuracy": epoch_metrics["accuracy"],
            "val_f1_macro": epoch_metrics["f1_macro"],
            "val_map@3": epoch_metrics["map@3"],
        })

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "accuracy": epoch_metrics["accuracy"],
            "f1_macro": epoch_metrics["f1_macro"],
            "map@3": epoch_metrics["map@3"]
        })
        best_map3 = max(best_map3, epoch_metrics["map@3"])

    return history, best_map3, final_val_preds


run = wandb.init(
    project=WANDB_PROJECT,
    name="bilstm-self-attention",
    job_type="train",
    config={
        "model": "bilstm-attention", "vocab_size": VOCAB_SIZE, "max_len": MAX_LEN,
        "emb_dim": 128, "hidden_dim": 256, "num_layers": 2, "dropout": 0.3,
        "epochs": 12, "batch_size": 64, "lr": 3e-4
    }
)

bilstm = BiLSTMModel(VOCAB_SIZE).to(DEVICE)
num_params = sum(p.numel() for p in bilstm.parameters() if p.requires_grad)
print(f"BiLSTM parameter count: {num_params:,}")
wandb.config.update({"num_params": num_params})

bilstm_history, bilstm_best_map3, bilstm_val_preds = run_training(
    bilstm, train_sub, val_sub, epochs=12, name="bilstm"
)

bilstm_metrics = compute_metrics(val_true, bilstm_val_preds)
print(f"\nRun 3 (Bi-LSTM + Attention) final -> accuracy: {bilstm_metrics['accuracy']:.4f} | "
      f"F1 (macro): {bilstm_metrics['f1_macro']:.4f} | MAP@3: {bilstm_metrics['map@3']:.4f}")

wandb.summary.update(bilstm_metrics)
all_run_results.append({
    "run": "bilstm-self-attention",
    "accuracy": bilstm_metrics["accuracy"],
    "f1_macro": bilstm_metrics["f1_macro"],
    "map@3": bilstm_metrics["map@3"]
})
wandb.finish()

## 10. Model Comparison & Results

Validation results across all models are aggregated and plotted to compare Accuracy, Macro F1-score, and MAP@3 performance.

In [ ]:
comparison_df = pd.DataFrame(all_run_results).set_index("run")
print(comparison_df)

comparison_df[["accuracy", "f1_macro", "map@3"]].plot(kind="bar", figsize=(9, 5))
plt.title("Model Comparison: Accuracy vs. Macro F1 vs. MAP@3")
plt.ylabel("Score")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

best_run = comparison_df["map@3"].idxmax()
print(f"\nBest run by MAP@3: {best_run}")

## 11. Test Predictions & Submission

The highest-performing model (Bi-LSTM with Self-Attention) is evaluated on the test set (`test.csv`). Ranked top-3 predictions per question are exported to `submission.csv`.

In [ ]:
print("Generating predictions on the test set using the Bi-LSTM model...")


def get_neural_probs(model, df, batch_size=64):
    model.eval()
    loader = DataLoader(MCQDataset(df), batch_size=batch_size, shuffle=False)
    all_probs = []
    with torch.no_grad():
        for options, _ in loader:
            probs = torch.softmax(model(options.to(DEVICE)), dim=-1).cpu().numpy()
            all_probs.append(probs)
    all_probs = np.concatenate(all_probs)
    
    result = []
    for i in range(len(all_probs)):
        prob_dict = {}
        for j, label in enumerate(LABELS):
            prob_dict[label] = float(all_probs[i, j])
        result.append(prob_dict)
    return result


test_probs = get_neural_probs(bilstm, test_df)

bilstm_predictions = []
for i in range(len(test_df)):
    probs_dict = test_probs[i]
    sorted_labels = sorted(LABELS, key=lambda label: probs_dict[label], reverse=True)
    bilstm_predictions.append(sorted_labels[:3])

os.makedirs('../submissions', exist_ok=True)
id_col = 'id' if 'id' in test_df.columns else 'ID'

submission = format_submission(
    ids=test_df[id_col].tolist(),
    predictions=bilstm_predictions,
    output_path='submission.csv'
)

print("\nSample predictions:")
print(submission.head(10).to_string(index=False))

## 12. Summary & Conclusion

Three models were constructed and evaluated:
- **TF-IDF + KNN**: Simple retrieval baseline.
- **TF-IDF + Logistic Regression**: Multi-class linear classifier.
- **Bi-LSTM + Self-Attention**: Deep contextual neural network yielding highest validation MAP@3.

Final predictions from the Bi-LSTM model are exported in `submission.csv`.